# 1 · Building a quasi-agent: generate → document → test

This notebook chains **three prompts** to MiniCPM (`openbmb/MiniCPM5-2B`) to turn a one-line description into a documented, tested Python function:

1. **Generate** – the user describes a function; the model writes it.
2. **Document** – the model adds a full docstring (description, parameters, return value, examples, edge cases).
3. **Test** – the model writes `unittest` test cases, and everything is saved to a `.py` file.

**Why "quasi"-agent?** The model keeps *memory*: each prompt and answer is appended to the same `messages` list, so every step sees the previous ones. But the sequence of steps is **hard-coded**: the model never decides what to do next, uses tools, or checks its own work. Those are the parts a real agent adds.

**Requirements:** same as notebook `0-test_MiniCPM` (`torch`, `transformers`, `accelerate`; in Colab run `!pip install -q transformers accelerate` first; a GPU is recommended). Step 1 asks for **keyboard input**, so run the cells one by one.

## 1. Imports

In [ ]:
import json                       # handy for passing structured data in prompts
from typing import Dict, List     # type hints for the chat messages

import torch                      # deep-learning backend used by the model
from transformers import AutoModelForCausalLM, AutoTokenizer

## 2. Load the model

Downloads MiniCPM from the Hugging Face Hub on first run (~5 GB, cached afterwards). `torch_dtype="auto"` keeps the precision the weights were saved in; `device_map="auto"` uses the GPU when available.

In [ ]:
# Model to test: MiniCPM, a ~2-billion-parameter chat model from OpenBMB
model_id = "openbmb/MiniCPM5-2B"

# The tokenizer converts text <-> token IDs and knows the model's chat format
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",   # keep the precision the weights were saved in
    device_map="auto",    # use the GPU automatically when available
)

print("Model loaded successfully!")

## 3. Helper functions

- `generate_response()` sends the whole conversation to the model and returns only its new reply (same helper as notebook 0, but with room for **1024** new tokens so that full functions and test suites fit).
- `extract_code_block()` pulls the code out of a Markdown-formatted answer, dropping the model's explanations around it.

In [ ]:
def generate_response(messages: List[Dict]) -> str:
    """Send a chat conversation to MiniCPM and return the model's reply.

    Args:
        messages: The conversation so far, as a list of dicts with a
            ``"role"`` ("system", "user" or "assistant") and a ``"content"``.

    Returns:
        The model's reply as plain text (without the prompt).
    """
    # 1. Format the conversation with the model's chat template and convert it
    #    to token IDs (PyTorch tensors) on the same device as the model.
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,   # add the "assistant's turn" marker
        enable_thinking=False,        # MiniCPM option: skip the hidden reasoning step
        return_dict=True,             # return input_ids and attention_mask
        return_tensors="pt",          # as PyTorch tensors
    ).to(model.device)

    # 2. Generate the answer (no gradients needed: we are not training).
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,      # enough for a function + docs or a test suite
            do_sample=False,          # greedy decoding: same input -> same output
        )

    # 3. Keep only the newly generated tokens (after the prompt) and decode them.
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,     # drop markers like <|im_end|>
    )

    return response

In [ ]:
def extract_code_block(response: str) -> str:
    """Return the code inside the first Markdown code block of a response.

    LLMs usually answer like: some text, then ```python ... ```, then more text.
    Splitting on the triple backticks gives
    [text before, code block, text after, ...], so element [1] is the code.

    Args:
        response: The model's full answer.

    Returns:
        The code without the surrounding backticks and the "python" language
        tag. If the response has no code block, it is returned unchanged.

    Note:
        Only the *first* code block is kept; anything in later blocks is lost.
    """
    # No code block at all: return the answer as it is
    if not '```' in response:
        return response

    # Take what is between the first pair of ``` markers
    code_block = response.split('```')[1].strip()

    # Remove the language tag that follows the opening backticks (```python)
    if code_block.startswith("python"):
        code_block = code_block[6:]   # len("python") == 6

    return code_block

## 4. Start the conversation

The `messages` list is the agent's **memory**. It starts with a system message that sets the model's role; every prompt and answer below is appended to it.

> Re-run this cell before starting a new function, otherwise the old conversation is kept.

In [ ]:
# The conversation history, shared by all three steps
messages = [
    {"role": "system", "content": "You are an expert software engineer that can generate any Python function."}
]

## 5. Step 1 – Generate the function

Asks you to describe a function (e.g. *sum up two integers*), sends the request to the model and keeps only the code from its answer.

In [ ]:
# Prompt 1: ask the user what function they want to create
print("\nWhat kind of function would you like to create?")
print("Example: 'A function that calculates the factorial of a number'")
print("Your description: ", end='')
function_description = input().strip()

# Ask the LLM to write a basic Python function based on the user's description
messages.append({"role": "user", "content": f"Write a Python function that {function_description}"})

# Store the response for use in the next prompts
initial_function = generate_response(messages)

# Separate the code from the model's commentary
initial_function = extract_code_block(initial_function)

print("\n=== Initial Function ===")
print(initial_function)

## 6. Step 2 – Add documentation

The model's own code is added to the conversation as an **assistant** message, so it "remembers" what it wrote. Then it is asked to document that function.

In [ ]:
# Prompt 2: add the code from step 1 to the conversation as the model's answer
messages.append({"role": "assistant", "content": initial_function})

# Ask the LLM to add comprehensive documentation
messages.append({
    "role": "user",
    "content": "Add comprehensive documentation of this function including description, parameters, "
               "return value, examples, and edge cases. Output the function in a ```python code block```."
})

documented_function = generate_response(messages)
documented_function = extract_code_block(documented_function)

print("\n=== Documented Function ===")
print(documented_function)

# Add the documented version to the conversation (wrapped back in a code block)
messages.append({"role": "assistant", "content": "```python\n\n" + documented_function + "\n\n```"})

## 7. Step 3 – Write tests and save to a file

The model writes `unittest` tests for the documented function. The function and the tests are then saved together in a `.py` file named after your description (e.g. `sum_up_two_integers.py`).

**Known limitation:** the model sometimes returns only the tests, sometimes the tests *and* the function again, and may forget `import unittest`, so the saved file may need small fixes before it runs. Handling this kind of unreliability is what real agents (with checks and retries) are for.

> In Colab the file is saved in the runtime (`/content`), which is wiped when the session ends. Download it from the 📁 *Files* panel to keep it.

In [ ]:
# Prompt 3: add test cases
messages.append({
    "role": "user",
    "content": "Add unittest test cases for this function, including tests for basic functionality, "
               "edge cases, error cases, and various input scenarios. Output the code in a ```python code block```."
})
test_cases = generate_response(messages)

# The model may output JUST the test cases or the test cases AND the code,
# so the extracted block can vary from run to run.
test_cases = extract_code_block(test_cases)
print("\n=== Test Cases ===")
print(test_cases)

# Build a file name from the description:
# lowercase -> keep only letters, digits and spaces -> spaces to underscores,
# max 30 characters, plus .py  (e.g. "sum up two integers" -> "sum_up_two_integers.py")
filename = function_description.lower()
filename = ''.join(c for c in filename if c.isalnum() or c.isspace())
filename = filename.replace(' ', '_')[:30] + '.py'

# Save the documented function followed by its tests
with open(filename, 'w') as f:
    f.write(documented_function + '\n\n' + test_cases)

print(f"\nSaved to {filename}")